Standardize timestamps

In [59]:
import pandas as pd

news_df = pd.read_csv(r"C:\sentiment_analysis\raw_analyst_ratings.csv")
stock_df = pd.read_csv(r"C:\sentiment_analysis\Data\AAPL.csv")
stock_df = pd.read_csv(r"C:\sentiment_analysis\Data\AMZN.csv")
stock_df = pd.read_csv(r"C:\sentiment_analysis\Data\GOOG.csv")
stock_df = pd.read_csv(r"C:\sentiment_analysis\Data\META.csv")
stock_df = pd.read_csv(r"C:\sentiment_analysis\Data\NVDA.csv")

print(news_df.columns)
news_df["date"] = pd.to_datetime(news_df["date"], errors="coerce")
news_df = news_df.dropna(subset=['date'])
news_df['date'] = pd.to_datetime(news_df['date'], utc=True).dt.tz_convert(None)
print(stock_df.columns)
stock_df["Date"] = pd.to_datetime(stock_df["Date"]).dt.tz_localize(None)

Index(['Unnamed: 0', 'headline', 'url', 'publisher', 'date', 'stock'], dtype='str')
Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='str')


create sentiment

In [61]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

news_df['sentiment_score'] = news_df['headline'].apply(
    lambda x: analyzer.polarity_scores(str(x))['compound']
)

Align news to trading days

In [62]:
stock_df = stock_df.sort_values('Date')
trading_days = stock_df['Date'].unique()
trading_days = pd.to_datetime(trading_days).tz_localize(None)

Mapping function (forward alignment)

In [65]:
def align_to_trading_day(ts, trading_days):
    if pd.isna(ts):
        return None
    ts = ts.normalize()
    idx = trading_days.searchsorted(ts)
    if idx < len(trading_days):
        return trading_days[idx]
    return None

In [66]:
news_df['trading_day'] = news_df['date'].apply(
    lambda x: align_to_trading_day(x, trading_days)
)

Aggregate daily sentiment

In [67]:
daily_sentiment = (
    news_df.groupby('trading_day')['sentiment_score']
    .mean()
    .reset_index()
)

Compute Stock Returns

In [ ]:
import numpy as np

stock_df = stock_df.sort_values('Date')

stock_df['log_return'] = np.log(stock_df['Close']).diff()

Merge datasets

In [71]:
import numpy as np

stock_df = stock_df.sort_values('Date')

stock_df['log_return'] = np.log(stock_df['Close']).diff()

merged = pd.merge(
    stock_df[['Date', 'log_return']],
    daily_sentiment,
    left_on='Date',
    right_on='trading_day',
    how='left'
)
merged['sentiment_score'] = merged['sentiment_score'].fillna(0)

Correlation analysis

In [72]:
from scipy.stats import pearsonr

corr, p = pearsonr(
    merged['sentiment_score'],
    merged['log_return']
)

print("Pearson correlation:", corr)
print("p-value:", p)

Pearson correlation: nan
p-value: nan


Spearman correlation (rank-based)

In [73]:
from scipy.stats import spearmanr

corr, p = spearmanr(
    merged['sentiment_score'],
    merged['log_return']
)

print("Spearman correlation:", corr)

Spearman correlation: nan
